# 03b — Preparação de Dataset Anual (Clima → Produção) | Vale do São Francisco

## Objetivo
Agregar **features climáticas mensais** para **anuais** e fazer **merge** com dados de produção **IBGE/PAM** (anuais).

## Por que este notebook existe?
Os dados do IBGE/PAM são **anuais** (1 registro por **ano × município × produto**).  
Logo, *replicar a produção anual para 12 meses* (como em abordagens mensais artificiais) **não** é apropriado para modelagem.

## Saída
- `data/processed/dataset_ml_anual.csv`

## Regras de Qualidade (importante)
- Split temporal e modelagem acontecem no **notebook 04**.
- Este notebook **não** deve incluir variáveis com **data leakage** (ex.: área colhida, rendimento, preço).


In [7]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 1) Configuração de caminhos

Boas práticas:
- Usar caminhos **relativos ao repositório** (funciona local, Colab, VSCode).
- Centralizar paths em um só lugar.

> Se você estiver no Colab, pode ajustar `DATA_DIR` para o local onde os CSVs estão.


In [8]:
# Raiz do projeto (ajuste se necessário)
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# Entradas (ajuste conforme seus arquivos reais)
PATH_CLIMA_MENSAL = PROCESSED_DIR / "clima_mensal_consolidado.csv"
PATH_PRODUCAO_ANUAL = PROCESSED_DIR / "pam_censo_agro_integrado_v2.csv"

# Saída
OUTPUT_PATH = PROCESSED_DIR / "dataset_ml_anual.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PATH_CLIMA_MENSAL:", PATH_CLIMA_MENSAL)
print("PATH_PRODUCAO_ANUAL:", PATH_PRODUCAO_ANUAL)
print("OUTPUT_PATH:", OUTPUT_PATH)


PROJECT_ROOT: /content
PATH_CLIMA_MENSAL: /content/data/processed/clima_mensal_consolidado.csv
PATH_PRODUCAO_ANUAL: /content/data/processed/pam_censo_agro_integrado_v2.csv
OUTPUT_PATH: /content/data/processed/dataset_ml_anual.csv


## 2) Carregar dados

- `df_clima_mensal`: clima mensal consolidado (idealmente gerado no notebook 02/03)
- `df_producao`: produção IBGE/PAM (anual)

Dica: se der erro de arquivo não encontrado, confira os caminhos na seção anterior.


In [9]:
def read_csv_checked(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {path}\n"
            f"Dica: ajuste os caminhos na seção 'Configuração de caminhos'."
        )
    return pd.read_csv(path, **kwargs)

df_clima_mensal = read_csv_checked(PATH_CLIMA_MENSAL)
df_producao = read_csv_checked(PATH_PRODUCAO_ANUAL)

print("Clima mensal:", df_clima_mensal.shape)
print("Produção anual:", df_producao.shape)

df_clima_mensal.head(3)


Clima mensal: (288, 9)
Produção anual: (48, 10)


,municipio,ano,mes,temp_media_mensal,temp_maxima_mensal,temp_minima_mensal,precipitacao_acumulada_mm,eto_acumulada_mm,vento_medio_mensal
0,Juazeiro,2013,1,28.264516,36.1,22.5,65.0,191.02,18.980645
1,Juazeiro,2013,2,29.596429,36.8,22.3,0.1,230.28,23.467857
2,Juazeiro,2013,3,29.554839,38.4,22.3,18.2,218.93,20.067742


## 3) Normalização e contrato mínimo de colunas

### Padronização de município
Remove sufixos entre parênteses e padroniza para UPPERCASE.

### Contrato mínimo (schema)
Falhar cedo ajuda a evitar bugs silenciosos no merge.


In [11]:
# ==============================================================================
# 🆕 ETAPA ADICIONAL: Recriar Features Agronômicas (Feature Engineering)
# ==============================================================================

print("⚙️ Recalculando features agronômicas mensais...")

# 1. GDD (Growing Degree Days) - Base 10°C
def calcular_gdd(temp_media, temp_base=10):
    return np.maximum(temp_media - temp_base, 0)

df_clima_mensal['gdd_mensal'] = calcular_gdd(df_clima_mensal['temp_media_mensal'])

# 2. Déficit Hídrico (P - ETO)
df_clima_mensal['deficit_hidrico_mm'] = (
    df_clima_mensal['precipitacao_acumulada_mm'] -
    df_clima_mensal['eto_acumulada_mm']
)

# 3. Índice de Stress Hídrico (0 a 100)
# Se ETO > 0: (1 - P/ETO) * 100. Se ETO=0, stress é 0.
df_clima_mensal['indice_stress_hidrico'] = np.where(
    df_clima_mensal['eto_acumulada_mm'] > 0,
    (1 - df_clima_mensal['precipitacao_acumulada_mm'] / df_clima_mensal['eto_acumulada_mm']) * 100,
    0
)
df_clima_mensal['indice_stress_hidrico'] = df_clima_mensal['indice_stress_hidrico'].clip(0, 100)

# 4. Amplitude Térmica
df_clima_mensal['amplitude_termica'] = (
    df_clima_mensal['temp_maxima_mensal'] -
    df_clima_mensal['temp_minima_mensal']
)

# 5. Estimativa de Dias com Temperatura Ótima (20-30°C)
def estimar_dias_temp_otima(temp_media, temp_min, temp_max, dias_mes=30,
                             temp_ideal_min=20, temp_ideal_max=30):
    # Desvio padrão estimado (Range / 4)
    std = (temp_max - temp_min) / 4.0

    if std == 0:
        return dias_mes if temp_ideal_min <= temp_media <= temp_ideal_max else 0

    # Probabilidade de estar no range ideal (CDF Normal)
    prob_ideal = (
        norm.cdf(temp_ideal_max, loc=temp_media, scale=std) -
        norm.cdf(temp_ideal_min, loc=temp_media, scale=std)
    )
    return prob_ideal * dias_mes

df_clima_mensal['dias_temp_otima_est'] = df_clima_mensal.apply(
    lambda row: estimar_dias_temp_otima(
        row['temp_media_mensal'],
        row['temp_minima_mensal'],
        row['temp_maxima_mensal']
    ),
    axis=1
)

print("✅ Features recriadas com sucesso!")
print(f"📋 Novas colunas: {['gdd_mensal', 'deficit_hidrico_mm', 'indice_stress_hidrico', 'amplitude_termica', 'dias_temp_otima_est']}")

⚙️ Recalculando features agronômicas mensais...
✅ Features recriadas com sucesso!
📋 Novas colunas: ['gdd_mensal', 'deficit_hidrico_mm', 'indice_stress_hidrico', 'amplitude_termica', 'dias_temp_otima_est']


In [12]:
import re

def standardize_municipio(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.replace(r"\s*\(.*\)\s*", "", regex=True)  # remove "(...)" e espaços ao redor
        .str.strip()
        .str.upper()
    )

# Padronizar municípios (se existirem)
for df_name, df_ in [("clima", df_clima_mensal), ("producao", df_producao)]:
    if "municipio" not in df_.columns:
        raise ValueError(f"Coluna 'municipio' ausente em df_{df_name}.")
    df_["municipio"] = standardize_municipio(df_["municipio"])

REQUIRED_CLIMA = {
    "municipio", "ano",
    "temp_media_mensal", "temp_maxima_mensal", "temp_minima_mensal",
    "precipitacao_acumulada_mm", "eto_acumulada_mm",
    "vento_medio_mensal",
    "gdd_mensal", "deficit_hidrico_mm", "indice_stress_hidrico",
    "amplitude_termica", "dias_temp_otima_est",
}

REQUIRED_PRODUCAO = {"municipio", "ano", "produto", "quantidade_produzida_t"}

missing_clima = REQUIRED_CLIMA - set(df_clima_mensal.columns)
missing_prod = REQUIRED_PRODUCAO - set(df_producao.columns)

if missing_clima:
    raise ValueError(f"Colunas ausentes no clima mensal: {sorted(missing_clima)}")
if missing_prod:
    raise ValueError(f"Colunas ausentes na produção anual: {sorted(missing_prod)}")

print("✅ Contrato mínimo validado (clima e produção).")


✅ Contrato mínimo validado (clima e produção).


## 4) Agregar clima mensal → anual

### Estratégia de agregação
- Temperaturas:
  - `temp_media_mensal` → **média anual**
  - `temp_maxima_mensal` → **máxima anual**
  - `temp_minima_mensal` → **mínima anual**
- Água (acumulados): precipitação e ETo → **soma anual**
- Vento: **média anual**
- Features engenheiradas:
  - `gdd_mensal` → soma anual
  - `deficit_hidrico_mm` → soma anual
  - `indice_stress_hidrico` → média anual
  - `amplitude_termica` → média anual
  - `dias_temp_otima_est` → soma anual


In [13]:
AGG_DICT = {
    # Temperaturas
    "temp_media_mensal": "mean",
    "temp_maxima_mensal": "max",
    "temp_minima_mensal": "min",
    # Água
    "precipitacao_acumulada_mm": "sum",
    "eto_acumulada_mm": "sum",
    # Vento
    "vento_medio_mensal": "mean",
    # Features engenheiradas
    "gdd_mensal": "sum",
    "deficit_hidrico_mm": "sum",
    "indice_stress_hidrico": "mean",
    "amplitude_termica": "mean",
    "dias_temp_otima_est": "sum",
}

df_clima_anual = (
    df_clima_mensal
    .groupby(["municipio", "ano"], as_index=False)
    .agg(AGG_DICT)
)

RENAME_MAP = {
    "temp_media_mensal": "temp_media_anual",
    "temp_maxima_mensal": "temp_max_anual",
    "temp_minima_mensal": "temp_min_anual",
    "precipitacao_acumulada_mm": "precipitacao_total_anual_mm",
    "eto_acumulada_mm": "eto_total_anual_mm",
    "vento_medio_mensal": "vento_medio_anual",
    "gdd_mensal": "gdd_total_anual",
    "deficit_hidrico_mm": "deficit_hidrico_total_anual_mm",
    "indice_stress_hidrico": "indice_stress_medio_anual",
    "amplitude_termica": "amplitude_termica_media_anual",
    "dias_temp_otima_est": "dias_temp_otima_total_anual",
}
df_clima_anual = df_clima_anual.rename(columns=RENAME_MAP)

print("✅ Clima anual:", df_clima_anual.shape)
print("Período:", df_clima_anual["ano"].min(), "-", df_clima_anual["ano"].max())

df_clima_anual.head(5)


✅ Clima anual: (24, 13)
Período: 2013 - 2024


,municipio,ano,temp_media_anual,temp_max_anual,temp_min_anual,precipitacao_total_anual_mm,eto_total_anual_mm,vento_medio_anual,gdd_total_anual,deficit_hidrico_total_anual_mm,indice_stress_medio_anual,amplitude_termica_media_anual,dias_temp_otima_total_anual
0,JUAZEIRO,2013,27.137606,38.4,16.5,311.6,2276.95,22.448316,205.651267,-1965.35,84.876705,15.416667,255.787382
1,JUAZEIRO,2014,26.353959,37.2,17.9,289.6,2158.91,22.974942,196.247512,-1869.31,85.611794,14.800000,276.633543
2,JUAZEIRO,2015,27.273674,37.8,17.3,270.3,2344.09,22.498347,207.284094,-2073.79,87.760188,14.891667,254.240278
3,JUAZEIRO,2016,27.321821,37.8,17.9,265.4,2343.42,22.277181,207.861846,-2078.02,89.710466,15.583333,247.708596
4,JUAZEIRO,2017,26.695780,36.5,17.4,309.0,2190.14,24.709351,200.349355,-1881.14,85.639619,14.175000,261.299015


## 5) Merge clima anual + produção anual

- Merge **inner** garante que você só mantém combinações onde há clima e produção.
- Após o merge, validamos missing values.


In [14]:
df_ml_anual = pd.merge(
    df_producao,
    df_clima_anual,
    on=["municipio", "ano"],
    how="inner",
)

print("✅ Merge concluído:", df_ml_anual.shape)
print("Período:", df_ml_anual["ano"].min(), "-", df_ml_anual["ano"].max())
print("Municípios:", df_ml_anual["municipio"].nunique())
print("Produtos:", df_ml_anual["produto"].nunique())

missing = df_ml_anual.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing):
    print("\n⚠️ Missing values (top 20):")
    display(missing.head(20))
else:
    print("\n✅ Sem missing values após merge.")

df_ml_anual.head(5)


✅ Merge concluído: (48, 21)
Período: 2013 - 2024
Municípios: 2
Produtos: 2

✅ Sem missing values após merge.


,municipio,ano,produto,area_colhida_ha,quantidade_produzida_t,rendimento_medio_kg_ha,preco_mediano_r$_kg,preco_medio_r$_kg,preco_std_r$_kg,num_observacoes_preco,temp_media_anual,temp_max_anual,temp_min_anual,precipitacao_total_anual_mm,eto_total_anual_mm,vento_medio_anual,gdd_total_anual,deficit_hidrico_total_anual_mm,indice_stress_medio_anual,amplitude_termica_media_anual,dias_temp_otima_total_anual
0,JUAZEIRO,2013,Manga,8086,210236,26000,1.365,1.365000,0.417193,2,27.137606,38.4,16.5,311.6,2276.95,22.448316,205.651267,-1965.35,84.876705,15.416667,255.787382
1,JUAZEIRO,2013,Uva,1270,27940,22000,4.300,4.443571,1.355959,14,27.137606,38.4,16.5,311.6,2276.95,22.448316,205.651267,-1965.35,84.876705,15.416667,255.787382
2,JUAZEIRO,2014,Manga,2130,55380,26000,1.255,1.445000,0.610710,4,26.353959,37.2,17.9,289.6,2158.91,22.974942,196.247512,-1869.31,85.611794,14.800000,276.633543
3,JUAZEIRO,2014,Uva,1576,39400,25000,4.335,4.574167,1.493017,12,26.353959,37.2,17.9,289.6,2158.91,22.974942,196.247512,-1869.31,85.611794,14.800000,276.633543
4,JUAZEIRO,2015,Manga,2130,44730,21000,1.590,1.706667,0.642988,3,27.273674,37.8,17.3,270.3,2344.09,22.498347,207.284094,-2073.79,87.760188,14.891667,254.240278


## 6) Seleção de colunas finais (sem leakage)

Regra de ouro: manter apenas variáveis que você conhece **antes** de observar a produção.

✅ Mantemos:
- IDs: `ano`, `municipio`, `produto`
- Target: `quantidade_produzida_t`
- Features climáticas anuais

❌ Não entram aqui (mesmo que existam no df_producao):
- `area_colhida_ha`
- `rendimento_medio_kg_ha`
- `preco_medio_r$_kg`


In [15]:
FINAL_COLUMNS = [
    # Identificadores
    "ano", "municipio", "produto",
    # Target
    "quantidade_produzida_t",
    # Features climáticas anuais
    "temp_media_anual",
    "temp_max_anual",
    "temp_min_anual",
    "precipitacao_total_anual_mm",
    "eto_total_anual_mm",
    "vento_medio_anual",
    "gdd_total_anual",
    "deficit_hidrico_total_anual_mm",
    "indice_stress_medio_anual",
    "amplitude_termica_media_anual",
    "dias_temp_otima_total_anual",
]

missing_cols = set(FINAL_COLUMNS) - set(df_ml_anual.columns)
if missing_cols:
    raise ValueError(f"Colunas finais ausentes no dataset merged: {sorted(missing_cols)}")

df_final = df_ml_anual[FINAL_COLUMNS].copy()

print("✅ Dataset final:", df_final.shape)
df_final.head(5)


✅ Dataset final: (48, 15)


,ano,municipio,produto,quantidade_produzida_t,temp_media_anual,temp_max_anual,temp_min_anual,precipitacao_total_anual_mm,eto_total_anual_mm,vento_medio_anual,gdd_total_anual,deficit_hidrico_total_anual_mm,indice_stress_medio_anual,amplitude_termica_media_anual,dias_temp_otima_total_anual
0,2013,JUAZEIRO,Manga,210236,27.137606,38.4,16.5,311.6,2276.95,22.448316,205.651267,-1965.35,84.876705,15.416667,255.787382
1,2013,JUAZEIRO,Uva,27940,27.137606,38.4,16.5,311.6,2276.95,22.448316,205.651267,-1965.35,84.876705,15.416667,255.787382
2,2014,JUAZEIRO,Manga,55380,26.353959,37.2,17.9,289.6,2158.91,22.974942,196.247512,-1869.31,85.611794,14.800000,276.633543
3,2014,JUAZEIRO,Uva,39400,26.353959,37.2,17.9,289.6,2158.91,22.974942,196.247512,-1869.31,85.611794,14.800000,276.633543
4,2015,JUAZEIRO,Manga,44730,27.273674,37.8,17.3,270.3,2344.09,22.498347,207.284094,-2073.79,87.760188,14.891667,254.240278


## 7) Validações rápidas (sanity checks)

Aqui a ideia é detectar problemas óbvios:
- distribuição por produto
- quantidade de registros por (produto, município)
- estatísticas do target


In [16]:
# Distribuição por produto
summary_prod = (
    df_final
    .groupby("produto")["quantidade_produzida_t"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(2)
)
summary_prod


,count,mean,std,min,max
produto,,,,,
Manga,24,242953.29,110451.22,44730,417566
Uva,24,172335.33,161114.26,27940,610739


In [17]:
# Registros por (produto, municipio)
count_by_seg = (
    df_final
    .groupby(["produto", "municipio"])["ano"]
    .nunique()
    .sort_values(ascending=False)
)
count_by_seg


produto  municipio
Manga    JUAZEIRO     12
         PETROLINA    12
Uva      JUAZEIRO     12
         PETROLINA    12
Name: ano, dtype: int64

In [19]:
df_final.describe(include="all")


,ano,municipio,produto,quantidade_produzida_t,temp_media_anual,temp_max_anual,temp_min_anual,precipitacao_total_anual_mm,eto_total_anual_mm,vento_medio_anual,gdd_total_anual,deficit_hidrico_total_anual_mm,indice_stress_medio_anual,amplitude_termica_media_anual,dias_temp_otima_total_anual
count,48.000000,48,48,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000
unique,NaN,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,JUAZEIRO,Manga,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,24,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2018.500000,NaN,NaN,207644.312500,26.809324,37.583333,17.300000,416.850000,2184.042917,22.565617,201.711889,-1767.192917,79.763625,14.873611,264.852488
std,3.488583,NaN,NaN,141230.167722,0.490451,0.707207,0.451946,166.875102,123.404954,0.792572,5.885413,279.240309,8.547209,0.426748,12.194992
min,2013.000000,NaN,NaN,27940.000000,25.750667,36.500000,16.500000,265.400000,1927.510000,21.423266,189.008003,-2078.450000,61.179840,14.175000,247.585322
25%,2015.750000,NaN,NaN,54212.500000,26.450005,37.200000,16.975000,298.900000,2139.080000,22.175002,197.400060,-1952.660000,76.645580,14.704167,255.332405
50%,2018.500000,NaN,NaN,178776.500000,26.926976,37.550000,17.350000,358.400000,2192.120000,22.362748,203.123710,-1839.630000,82.374246,14.875000,261.503570
75%,2021.250000,NaN,NaN,325092.000000,27.204997,37.900000,17.600000,466.850000,2271.475000,22.769560,206.459969,-1694.152500,85.628279,15.089583,276.849548


## 8) Salvar dataset anual

Gera:
- `data/processed/dataset_ml_anual.csv`


In [20]:
df_final.to_csv(OUTPUT_PATH, index=False)

size_kb = df_final.memory_usage(deep=True).sum() / 1024

print("✅ Dataset anual salvo:", OUTPUT_PATH)
print("Dimensões:", df_final.shape)
print(f"Tamanho em memória (aprox.): {size_kb:.2f} KB")

print("\n" + "=" * 70)
print("DATASET ANUAL PREPARADO COM SUCESSO ✅")
print("=" * 70)
print(f"Registros: {df_final.shape[0]} (ano × município × produto)")
print(f"Anos: {df_final['ano'].min()}–{df_final['ano'].max()} ({df_final['ano'].nunique()} anos)")
print(f"Municípios: {df_final['municipio'].nunique()}")
print(f"Produtos: {df_final['produto'].nunique()}")
print(f"Features climáticas: {len(FINAL_COLUMNS) - 4}")


✅ Dataset anual salvo: /content/data/processed/dataset_ml_anual.csv
Dimensões: (48, 15)
Tamanho em memória (aprox.): 10.18 KB

DATASET ANUAL PREPARADO COM SUCESSO ✅
Registros: 48 (ano × município × produto)
Anos: 2013–2024 (12 anos)
Municípios: 2
Produtos: 2
Features climáticas: 11
